# 01 — Synthetic BD MFS Transaction Data Generation

**Goal:** Generate 10,000+ realistic Bangladesh Mobile Financial Services (MFS) transactions
with 5 injected AML anomaly types, calibrated to BFIU reporting requirements.

---

| Parameter | Value |
|---|---|
| Total transactions | ~10,000 |
| Anomaly rate | ~5% (realistic for AML datasets) |
| Currency | BDT (Bangladeshi Taka) |
| Reporting threshold | BDT 10,000 (BFIU Rule 5) |
| Coverage period | Q1 2024 (Jan – Mar) |

## Bangladesh MFS Context

**bKash** (50M+ users) and **Nagad** (30M+ users) are the dominant Mobile Financial Services in Bangladesh.
Transactions use phone numbers (`01X-XXXXXXXX`) as account identifiers.

### Key BFIU Regulations
- **Circular No. 5 (2019):** MFS providers must report transactions ≥ BDT 10,000 within 3 working days
- **ML/TF Prevention Act 2012:** Structuring transactions to avoid the BDT 10,000 threshold is a criminal offence
- **Guideline 2020:** Dormant account reactivation with high-value transactions requires Enhanced Due Diligence (EDD)

### 5 Anomaly Types in This Dataset

| Anomaly | Pattern | BFIU Signal |
|---|---|---|
| `STRUCTURING` | Multiple txns just below BDT 10,000 within 24h | Deliberate threshold evasion |
| `VELOCITY` | ≥ 5 txns from same account within 60 minutes | Layering / account takeover |
| `LATE_NIGHT` | Transactions between 01:00–04:00 AM | Operational risk flag |
| `ROUND_AMOUNT` | Large exact round amounts (50k, 100k, 200k BDT) | Placement signal |
| `DORMANT_SPIKE` | Account inactive 30+ days → sudden high-value txn | EDD trigger |

In [ ]:
import os
import sys

os.chdir(os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

plt.style.use('dark_background')
for param in ['axes.facecolor', 'figure.facecolor', 'savefig.facecolor']:
    plt.rcParams[param] = '#0a0d16'
plt.rcParams['axes.edgecolor'] = '#181e30'
plt.rcParams['grid.color'] = '#181e30'

ACCENT = '#00c2f0'
WARN   = '#f5a623'
DANGER = '#e05a5a'
GREEN  = '#22d46a'

print('✓ Libraries loaded')
print(f'✓ Working directory: {os.getcwd()}')

## Step 1 — Generate Synthetic Transaction Data

In [ ]:
# Run the data generator — creates outputs/raw_transactions.csv
%run generate_data.py

In [ ]:
df = pd.read_csv('outputs/raw_transactions.csv', parse_dates=['timestamp'])
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(10)

## Step 2 — Dataset Overview

In [ ]:
print('=== Basic Stats ===')
print(f'Date range      : {df.timestamp.min().date()} → {df.timestamp.max().date()}')
print(f'Unique senders  : {df.sender_id.nunique():,}')
print(f'Unique receivers: {df.receiver_id.nunique():,}')
print(f'Amount range    : BDT {df.amount_bdt.min():,} → {df.amount_bdt.max():,}')
print(f'Mean amount     : BDT {df.amount_bdt.mean():,.0f}')
print(f'Median amount   : BDT {df.amount_bdt.median():,.0f}')
print()
print(df.dtypes)

## Step 3 — Anomaly Distribution

> **Note on class imbalance:** Real-world AML datasets have ~1-3% anomaly rate.
> Our 5% is intentionally slightly higher for better model training signal.
> Class imbalance must be handled explicitly in ML (Step 03 notebook).

In [ ]:
flag_counts = df['anomaly_flag'].value_counts()
flag_pct    = df['anomaly_flag'].value_counts(normalize=True) * 100

print('Anomaly breakdown:')
for flag in flag_counts.index:
    bar = '█' * int(flag_pct[flag])
    print(f'  {flag:<20} {flag_counts[flag]:>5,}  ({flag_pct[flag]:5.1f}%)  {bar}')

fig, ax = plt.subplots(figsize=(10, 4))
colors = [GREEN if f == 'NORMAL' else DANGER for f in flag_counts.index]
bars = ax.bar(flag_counts.index, flag_counts.values, color=colors, edgecolor='#181e30', linewidth=0.5)
ax.set_title('Transaction Count by Anomaly Type', color='white', pad=12, fontsize=13)
ax.set_ylabel('Count')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'{bar.get_height():,.0f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
os.makedirs('images', exist_ok=True)
plt.savefig('images/01_anomaly_dist.png', dpi=120, bbox_inches='tight')
plt.show()

## Step 4 — Transaction Type & Geographic Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

tx_counts = df['tx_type'].value_counts()
axes[0].barh(tx_counts.index, tx_counts.values, color=ACCENT, edgecolor='#181e30')
axes[0].set_title('Transaction Types', color='white', fontsize=12)
axes[0].set_xlabel('Count')
for i, (val, label) in enumerate(zip(tx_counts.values, tx_counts.index)):
    axes[0].text(val + 30, i, f'{val:,}', va='center', fontsize=9)

div_counts = df['division'].value_counts()
axes[1].barh(div_counts.index, div_counts.values, color=WARN, edgecolor='#181e30')
axes[1].set_title('By Division (Dhaka ~45% — largest urban MFS market)', color='white', fontsize=12)
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.savefig('images/01_tx_types.png', dpi=120, bbox_inches='tight')
plt.show()
print('cash_out dominates in BD MFS — typical for rent, utility bills, bazar.')

## Step 5 — Amount Distribution

> **BD MFS quirk:** Round amounts (500, 1000, 5000 BDT) are *normal* — not suspicious.
> `ROUND_AMOUNT` only flags amounts ≥ 50,000 BDT that are outliers vs the sender's own profile.
> This calibration separates this toolkit from generic AML tools.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(df['amount_bdt'], bins=80, color=ACCENT, edgecolor='#181e30', linewidth=0.3)
axes[0].set_title('Amount Distribution — Raw (BDT)', color='white', fontsize=12)
axes[0].set_xlabel('Amount (BDT)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
axes[0].axvline(x=10000, color=DANGER, linestyle='--', linewidth=1.5, label='BFIU threshold (10,000)')
axes[0].legend(fontsize=9)

axes[1].hist(np.log1p(df['amount_bdt']), bins=60, color=GREEN, edgecolor='#181e30', linewidth=0.3)
axes[1].set_title('Amount Distribution — log(1+x)', color='white', fontsize=12)
axes[1].set_xlabel('log(Amount + 1)')
axes[1].axvline(x=np.log1p(10000), color=DANGER, linestyle='--', linewidth=1.5, label='BFIU threshold')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('images/01_amounts.png', dpi=120, bbox_inches='tight')
plt.show()

print('Median amount by anomaly type:')
print(df.groupby('anomaly_flag')['amount_bdt'].agg(['median','mean','max']).to_string())

## Step 6 — Temporal Patterns

In [ ]:
df['hour']        = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.day_name()
df['date']        = df['timestamp'].dt.date

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

hourly = df.groupby('hour').size()
hour_colors = [DANGER if h in [1, 2, 3] else ACCENT for h in hourly.index]
axes[0].bar(hourly.index, hourly.values, color=hour_colors, edgecolor='#181e30', linewidth=0.3)
axes[0].axvspan(0.5, 3.5, alpha=0.12, color=DANGER, label='Late-night window')
axes[0].set_title('Transactions by Hour of Day', color='white', fontsize=12)
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Count')
axes[0].set_xticks(range(0, 24, 2))
axes[0].legend(fontsize=9)

daily = df.groupby('date').size()
axes[1].plot(daily.index, daily.values, color=ACCENT, linewidth=1)
axes[1].fill_between(daily.index, daily.values, alpha=0.2, color=ACCENT)
axes[1].set_title('Daily Transaction Volume (Q1 2024)', color='white', fontsize=12)
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Transactions per day')
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig('images/01_temporal.png', dpi=120, bbox_inches='tight')
plt.show()

## Summary

| Metric | Value |
|---|---|
| Total transactions | ~10,000 |
| Normal | ~9,500 (95%) |
| Anomalous | ~500 (5%) |
| Date range | Jan 1 – Mar 31, 2024 |
| Output | `outputs/raw_transactions.csv` |

**BD-realism decisions:** `cash_out` dominates (38%), round amounts normal, bKash phone format, Dhaka-weighted, BDT 10k BFIU threshold.

**Next:** `02_rule_engine.ipynb`